In [1]:
# ============================================================
# Tome un notebook que ya tenia que hacia algo similar se lo pase al claude y le dije oye compa cambiame esto a esto otro, mas alla de eso no lo revise mucho
# ============================================================
import os
import time
import requests
import pandas as pd
from pathlib import Path
import pyarrow

TMDB_TOKEN = os.getenv("TMDB_TOKEN", "TOKEN_AQUI")

HEADERS = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json",
}

CACHE_DIR = Path("./animation_cache")
CACHE_DIR.mkdir(exist_ok=True)

SHOWS_PARQUET = CACHE_DIR / "tmdb_shows.parquet"
IMDB_IDS_PARQUET = CACHE_DIR / "tmdb_imdb_ids.parquet"
RATINGS_FILE = CACHE_DIR / "title.ratings.tsv.gz"

YEAR_START = 1960
YEAR_END = 2026
ANIMATION_GENRE_ID = 16

In [2]:
# ============================================================
# Cell 2 — Pull animated shows from TMDb Discover (resumable + retries)
# ============================================================
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SHOWS_PROGRESS_PARQUET = CACHE_DIR / "tmdb_shows_progress.parquet"
SHOWS_DONE_YEARS = CACHE_DIR / "tmdb_shows_done_years.txt"

def make_session():
    s = requests.Session()
    retry = Retry(
        total=8,
        backoff_factor=1.5,        # 1.5s, 3s, 6s, 12s, ...
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update(HEADERS)
    return s

def load_done_years():
    if not SHOWS_DONE_YEARS.exists():
        return set()
    try:
        return set(int(x) for x in SHOWS_DONE_YEARS.read_text().split() if x.strip())
    except FileNotFoundError:
        return set()

def mark_year_done(year):
    with open(SHOWS_DONE_YEARS, "a") as f:
        f.write(f"{year}\n")

def fetch_year(session, year, max_attempts=5):
    """Fetch all pages for a given year, with manual retry on connection errors."""
    rows = []
    page = 1
    while True:
        for attempt in range(max_attempts):
            try:
                r = session.get(
                    "https://api.themoviedb.org/3/discover/tv",
                    params={
                        "with_genres": ANIMATION_GENRE_ID,
                        "first_air_date_year": year,
                        "page": page,
                        "sort_by": "popularity.desc",
                    },
                    timeout=20,
                )
                if r.status_code == 429:
                    time.sleep(3)
                    continue
                r.raise_for_status()
                data = r.json()
                break
            except (requests.ConnectionError, requests.Timeout) as e:
                wait = 2 ** attempt
                print(f"    year {year} page {page} attempt {attempt+1} failed ({e.__class__.__name__}), sleeping {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError(f"Year {year} page {page} failed after {max_attempts} attempts")

        for show in data.get("results", []):
            rows.append({
                "tmdb_id": show["id"],
                "name": show.get("name"),
                "first_air_date": show.get("first_air_date"),
                "origin_country": ",".join(show.get("origin_country", []) or []),
                "original_language": show.get("original_language"),
                "vote_average_tmdb": show.get("vote_average"),
                "vote_count_tmdb": show.get("vote_count"),
                "popularity": show.get("popularity"),
            })

        total_pages = min(data.get("total_pages", 1), 500)
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.05)
    return rows

def fetch_shows_resumable():
    # If final cache exists, just use it
    if SHOWS_PARQUET.exists():
        print(f"Loading cached shows from {SHOWS_PARQUET}")
        return pd.read_parquet(SHOWS_PARQUET)

    # Load any partial progress
    if SHOWS_PROGRESS_PARQUET.exists():
        progress = pd.read_parquet(SHOWS_PROGRESS_PARQUET)
        all_rows = progress.to_dict("records")
        print(f"Resuming with {len(all_rows)} rows already collected")
    else:
        all_rows = []

    done_years = load_done_years()
    session = make_session()

    for year in range(YEAR_START, YEAR_END + 1):
        if year in done_years:
            continue
        try:
            year_rows = fetch_year(session, year)
            all_rows.extend(year_rows)
            mark_year_done(year)

            # Checkpoint after every year
            pd.DataFrame(all_rows).drop_duplicates("tmdb_id").to_parquet(
                SHOWS_PROGRESS_PARQUET, index=False
            )
            print(f"  {year}: +{len(year_rows)} rows, cumulative {len(all_rows)}")
        except Exception as e:
            print(f"  {year}: FAILED ({e}) — rerun the cell to resume")
            raise

    df = pd.DataFrame(all_rows).drop_duplicates("tmdb_id").reset_index(drop=True)
    df.to_parquet(SHOWS_PARQUET, index=False)
    print(f"Saved {len(df)} shows to {SHOWS_PARQUET}")
    return df

shows = fetch_shows_resumable()
shows.head()

  1961: +14 rows, cumulative 14
  1962: +18 rows, cumulative 32
  1963: +17 rows, cumulative 49
  1964: +22 rows, cumulative 71
  1965: +30 rows, cumulative 101
  1966: +33 rows, cumulative 134
  1967: +43 rows, cumulative 177
  1968: +31 rows, cumulative 208
  1969: +41 rows, cumulative 249
  1970: +26 rows, cumulative 275
  1971: +30 rows, cumulative 305
  1972: +44 rows, cumulative 349
  1973: +48 rows, cumulative 397
  1974: +42 rows, cumulative 439
  1975: +45 rows, cumulative 484
  1976: +49 rows, cumulative 533
  1977: +67 rows, cumulative 600
  1978: +56 rows, cumulative 656
  1979: +52 rows, cumulative 708
  1980: +45 rows, cumulative 753
  1981: +63 rows, cumulative 816
  1982: +45 rows, cumulative 861
  1983: +79 rows, cumulative 940
  1984: +76 rows, cumulative 1016
  1985: +67 rows, cumulative 1083
  1986: +86 rows, cumulative 1169
  1987: +91 rows, cumulative 1260
  1988: +90 rows, cumulative 1350
  1989: +107 rows, cumulative 1457
  1990: +97 rows, cumulative 1554
  1991

,tmdb_id,name,first_air_date,origin_country,original_language,vote_average_tmdb,vote_count_tmdb,popularity
0,30773,The Yogi Bear Show,1961-01-30,US,en,7.059,247,8.4200
1,4232,Top Cat,1961-09-27,US,en,7.525,366,5.3113
2,20785,Points of View,1961-10-02,GB,en,0.000,0,3.0311
3,210220,Otogi Manga Calendar,1961-05-01,JP,ja,0.000,0,2.5034
4,551,Supercar,1961-01-28,GB,en,7.091,11,1.9342


In [7]:
# ============================================================
# Cell 3 — Fetch IMDb IDs (multithreaded, resumable)
# Esta la tenia hecha pero se demoraba mucho con estos datos asi que le dije al claudio que me lo cambiara a multithread
# ============================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

IMDB_IDS_PKL = CACHE_DIR / "tmdb_imdb_ids.pkl"  # adjust if you kept .parquet

MAX_WORKERS = 16              # TMDb tolerates ~50 req/sec; 16 threads is safe
CHECKPOINT_EVERY = 500

def fetch_one_imdb_id(session, tmdb_id, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            r = session.get(
                f"https://api.themoviedb.org/3/tv/{tmdb_id}/external_ids",
                timeout=15,
            )
            if r.status_code == 429:
                time.sleep(2 + attempt)
                continue
            if r.status_code == 404:
                return tmdb_id, None
            r.raise_for_status()
            return tmdb_id, r.json().get("imdb_id")
        except (requests.ConnectionError, requests.Timeout):
            time.sleep(2 ** attempt)
    return tmdb_id, None

def fetch_imdb_ids_parallel(shows_df):
    if IMDB_IDS_PKL.exists():
        cached = pd.read_pickle(IMDB_IDS_PKL)
    else:
        cached = pd.DataFrame(columns=["tmdb_id", "imdb_id"])

    done_ids = set(cached["tmdb_id"].tolist())
    todo = shows_df[~shows_df["tmdb_id"].isin(done_ids)]["tmdb_id"].tolist()
    print(f"{len(done_ids)} cached, {len(todo)} to fetch with {MAX_WORKERS} workers")

    if not todo:
        return cached

    session = make_session()  # from Cell 2; reuses connection pool, retries
    results = []
    lock = threading.Lock()
    completed = 0

    def checkpoint():
        combined = pd.concat(
            [cached, pd.DataFrame(results, columns=["tmdb_id", "imdb_id"])],
            ignore_index=True,
        ).drop_duplicates("tmdb_id")
        combined.to_pickle(IMDB_IDS_PKL)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(fetch_one_imdb_id, session, tid): tid for tid in todo}
        for fut in as_completed(futures):
            tmdb_id, imdb_id = fut.result()
            with lock:
                results.append((tmdb_id, imdb_id))
                completed += 1
                print(f"Completed {fut}")
                if completed % CHECKPOINT_EVERY == 0:
                    checkpoint()
                    print(f"  {completed}/{len(todo)} done, checkpointed")

    checkpoint()
    combined = pd.read_pickle(IMDB_IDS_PKL)
    print(f"Total IMDb IDs cached: {len(combined)}")
    return combined

imdb_ids = fetch_imdb_ids_parallel(shows)
imdb_ids.head()

0 cached, 13884 to fetch with 16 workers
Completed <Future at 0x7f3ae0cedc50 state=finished returned tuple>
Completed <Future at 0x7f3ae0cb22d0 state=finished returned tuple>
Completed <Future at 0x7f3b1e24bbd0 state=finished returned tuple>
Completed <Future at 0x7f3b047b9250 state=finished returned tuple>
Completed <Future at 0x7f3ba5163ad0 state=finished returned tuple>
Completed <Future at 0x7f3b047b9a50 state=finished returned tuple>
Completed <Future at 0x7f3b1c104350 state=finished returned tuple>
Completed <Future at 0x7f3ae0ced1d0 state=finished returned tuple>
Completed <Future at 0x7f3b047b9650 state=finished returned tuple>
Completed <Future at 0x7f3ae0ced650 state=finished returned tuple>
Completed <Future at 0x7f3b047b8e50 state=finished returned tuple>
Completed <Future at 0x7f3b047b9e50 state=finished returned tuple>
Completed <Future at 0x7f3b1e591650 state=finished returned tuple>
Completed <Future at 0x7f3b1e591d50 state=finished returned tuple>
Completed <Future at 

,tmdb_id,imdb_id
0,3571,tt0054527
1,210220,tt21115548
2,1842,tt0054514
3,15119,tt0281433
4,30773,tt0255768


In [8]:
# ============================================================
# Cell 4 — Load IMDb ratings
# Se explica solo
# ============================================================
def load_imdb_ratings():
    if not RATINGS_FILE.exists():
        print("Downloading IMDb ratings...")
        url = "https://datasets.imdbws.com/title.ratings.tsv.gz"
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()
        with open(RATINGS_FILE, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Saved to {RATINGS_FILE}")

    ratings = pd.read_csv(
        RATINGS_FILE,
        sep="\t",
        usecols=["tconst", "averageRating", "numVotes"],
    ).rename(columns={"tconst": "imdb_id"})
    print(f"Loaded {len(ratings):,} IMDb ratings")
    return ratings

ratings = load_imdb_ratings()
ratings.head()


Saved to animation_cache/title.ratings.tsv.gz
Loaded 1,680,211 IMDb ratings


,imdb_id,averageRating,numVotes
0,tt0000001,5.7,2211
1,tt0000002,5.4,319
2,tt0000003,6.4,2344
3,tt0000004,5.1,192
4,tt0000005,6.2,3059


In [9]:
# ============================================================
# Cell 5 — Join & clean
#Adaptacion bien literal de lo que ya tenia
# ============================================================
MIN_VOTES = 1000

df = (
    shows
    .merge(imdb_ids, on="tmdb_id", how="left")
    .merge(ratings, on="imdb_id", how="inner")
)

# Take the first listed country (simplification; revisit for co-prods later)
df["country"] = df["origin_country"].str.split(",").str[0]
df = df[df["country"].notna() & (df["country"] != "")]

df["year"] = pd.to_datetime(df["first_air_date"], errors="coerce").dt.year

df_filtered = df[df["numVotes"] >= MIN_VOTES].copy()
print(f"{len(df)} merged rows, {len(df_filtered)} after vote threshold")
df_filtered.head()



8579 merged rows, 2081 after vote threshold


,tmdb_id,name,first_air_date,origin_country,original_language,vote_average_tmdb,vote_count_tmdb,popularity,imdb_id,averageRating,numVotes,country,year
0,30773,The Yogi Bear Show,1961-01-30,US,en,7.059,247,8.4200,tt0255768,6.6,11939,US,1961
1,4232,Top Cat,1961-09-27,US,en,7.525,366,5.3113,tt0054572,7.1,8745,US,1961
9,2362,The Jetsons,1962-09-23,US,en,7.329,504,9.3683,tt0055683,7.0,26419,US,1962
11,15179,Bolek and Lolek,1962-08-12,PL,pl,7.300,29,2.4561,tt0146370,7.0,2114,PL,1962
22,10826,Astro Boy,1963-01-01,JP,ja,7.400,17,15.2785,tt0056739,7.4,1182,JP,1963


In [10]:
# ============================================================
# Cell 6 — Country ranking (Bayesian-weighted)
# ============================================================
MIN_SHOWS_PER_COUNTRY = 20
PRIOR_WEIGHT = 500  # m: tune higher to pull more toward global mean

C = df_filtered["averageRating"].mean()  # global mean rating

df_filtered["weighted_rating"] = (
    df_filtered["numVotes"] * df_filtered["averageRating"] + PRIOR_WEIGHT * C
) / (df_filtered["numVotes"] + PRIOR_WEIGHT)

ranking = (
    df_filtered.groupby("country")
    .agg(
        n_shows=("imdb_id", "count"),
        avg_rating=("averageRating", "mean"),
        weighted_rating=("weighted_rating", "mean"),
        total_votes=("numVotes", "sum"),
    )
    .query(f"n_shows >= {MIN_SHOWS_PER_COUNTRY}")
    .sort_values("weighted_rating", ascending=False)
    .round(3)
)

print(f"Global mean rating: {C:.3f}")
ranking

Global mean rating: 7.156


,n_shows,avg_rating,weighted_rating,total_votes
country,,,,
JP,1040,7.348,7.349,10717904
FR,34,7.326,7.302,151700
GB,58,7.197,7.183,206539
US,773,6.915,6.991,12450878
CA,83,6.551,6.692,364956


In [14]:
# ============================================================
# Cell 6b
# Le cambie las cantidades de votos y cosas minimas para que aparezcan mas en el mapa
# ============================================================
MIN_VOTES_MAP = 100         # 1000
MIN_SHOWS_MAP = 3           #  20
PRIOR_WEIGHT_MAP = 200      #  500 

df_map = df[df["numVotes"] >= MIN_VOTES_MAP].copy()
C_map = df_map["averageRating"].mean()

df_map["weighted_rating"] = (
    df_map["numVotes"] * df_map["averageRating"] + PRIOR_WEIGHT_MAP * C_map
) / (df_map["numVotes"] + PRIOR_WEIGHT_MAP)

ranking_map = (
    df_map.groupby("country")
    .agg(
        n_shows=("imdb_id", "count"),
        avg_rating=("averageRating", "mean"),
        weighted_rating=("weighted_rating", "mean"),
        total_votes=("numVotes", "sum"),
    )
    .query(f"n_shows >= {MIN_SHOWS_MAP}")
    .sort_values("weighted_rating", ascending=False)
    .round(3)
)


print(f"Loose ranking:  {len(ranking_map)} countries")
ranking_map.head(20)

Strict ranking: 5 countries
Loose ranking:  30 countries


,n_shows,avg_rating,weighted_rating,total_votes
country,,,,
HU,17,7.735,7.534,12086
MY,7,7.571,7.303,3672
SU,14,7.307,7.290,21374
BR,27,7.626,7.229,8876
CN,101,7.281,7.177,96042
FI,10,7.240,7.114,6249
XC,6,7.433,7.093,946
AR,4,7.075,7.061,1861
NL,4,6.900,7.037,2108


In [11]:
# ============================================================
# Cell 7 — Export results to readable files
# Claudio Original, es solo para guardar los archivos para subir al git, quedan en csv, o si quieren tambien en excel pero no me di la lata
# ============================================================
OUTPUT_DIR = Path("./animation_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# 1. Country ranking — the main result
ranking_out = ranking.reset_index()
ranking_out.to_csv(OUTPUT_DIR / "country_ranking.csv", index=False)

# 2. Full joined dataset — every show with country, year, ratings
shows_out = df_filtered[[
    "tmdb_id", "imdb_id", "name", "country", "origin_country",
    "year", "first_air_date", "original_language",
    "averageRating", "numVotes", "weighted_rating",
    "vote_average_tmdb", "vote_count_tmdb", "popularity",
]].sort_values(["country", "weighted_rating"], ascending=[True, False])
shows_out.to_csv(OUTPUT_DIR / "shows_full.csv", index=False)

# 4. Excel workbook with everything in one file (optional, nice for sharing)
try:
    with pd.ExcelWriter(OUTPUT_DIR / "animation_analysis.xlsx", engine="openpyxl") as xl:
        ranking_out.to_excel(xl, sheet_name="Country Ranking", index=False)
        shows_out.to_excel(xl, sheet_name="All Shows", index=False)
        ranking.sort_values("n_shows", ascending=False).reset_index().to_excel(
            xl, sheet_name="By Volume", index=False
        )
    print("Excel workbook written.")
except ImportError:
    print("openpyxl not installed — skipping .xlsx export. Run: pip install openpyxl")

print(f"\nFiles written to {OUTPUT_DIR.resolve()}:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

openpyxl not installed — skipping .xlsx export. Run: pip install openpyxl

Files written to /home/asefito/Documents/dato/animation_output:
  country_ranking.csv  (187 bytes)
  shows_full.csv  (220,345 bytes)


In [15]:
# ============================================================
# Cell 8 — Proportional symbol map
# ============================================================
# Mapa de simbolos proporcionales

import plotly.express as px
import pycountry

# ---- 1. Convert alpha-2 country codes to alpha-3 (Plotly needs ISO-3) ----
def alpha2_to_alpha3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except (AttributeError, KeyError):
        return None

def alpha2_to_name(code):
    try:
        return pycountry.countries.get(alpha_2=code).name
    except (AttributeError, KeyError):
        return code

map_df = ranking_map.reset_index().copy()
map_df["iso_alpha"] = map_df["country"].apply(alpha2_to_alpha3)
map_df["country_name"] = map_df["country"].apply(alpha2_to_name)
map_df = map_df.dropna(subset=["iso_alpha"])

# ---- 2. Build the bubble map ----
fig = px.scatter_geo(
    map_df,
    locations="iso_alpha",
    size="n_shows",                    # bubble size = volume
    color="weighted_rating",           # bubble color = quality
    hover_name="country_name",
    hover_data={
        "iso_alpha": False,
        "n_shows": True,
        "avg_rating": ":.2f",
        "weighted_rating": ":.2f",
        "total_votes": ":,",
    },
    color_continuous_scale="Viridis",
    size_max=60,                       # cap so US doesn't swallow the map
    projection="natural earth",
    title="Animated Shows by Country — Volume (size) & Rating (color)",
)

fig.update_geos(
    showcountries=True,
    countrycolor="rgba(120,120,120,0.4)",
    showland=True,
    landcolor="rgb(243, 243, 243)",
    showocean=True,
    oceancolor="rgb(220, 235, 245)",
)

fig.update_layout(
    height=650,
    margin=dict(l=0, r=0, t=50, b=0),
    coloraxis_colorbar=dict(title="Weighted<br>rating"),
)

fig.show()

# ---- 3. Export to standalone HTML ----
fig.write_html(OUTPUT_DIR / "animation_bubble_map.html", include_plotlyjs="cdn")
print(f"Saved to {OUTPUT_DIR / 'animation_bubble_map.html'}")

Saved to animation_output/animation_bubble_map.html
